# 06_07_EXO_differential_photometry  r_filter

## 필요한 모듈

이 프로젝트를 위해서는 아래의 모듈이 필요하다. 

> numpy, pandas, matplotlib, scipy, astropy, astroquery, photutils, ccdproc, version_information

### 모듈 설치

1. 콘솔 창에서 모듈을 설치할 때는 아래와 같은 형식으로 입력하면 된다.

>pip install module_name==version

>conda install module_name=version

2. 주피터 노트북(코랩 포함)에 설치 할 때는 아래의 셀을 실행해서 실행되지 않은 모듈을 설치할 수 있다. (pip 기준) 만약 아나콘다 환경을 사용한다면 7행을 콘다 설치 명령어에 맞게 수정하면 된다.

### 모듈 버전 확인

아래 셀을 실행하면 이 노트북을 실행한 파이썬 및 관련 모듈의 버전을 확인할 수 있다.

In [1]:
import importlib, sys, subprocess
packages = "numpy, pandas, matplotlib, scipy, astropy, astroquery, photutils, ysfitsutilpy, ysphotutilpy, ccdproc, aplpy, version_information" # required modules
pkgs = packages.split(", ")
for pkg in pkgs :
    if not importlib.util.find_spec(pkg):
        print(f"**** module {pkg} is being installed")
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg, '-q'])
    else: 
        print(f"**** module {pkg} is installed")

%load_ext version_information
import time
now = time.strftime("%Y-%m-%d %H:%M:%S (%Z = GMT%z)")
print(f"This notebook was generated at {now} ")

vv = %version_information {packages}
for i, pkg in enumerate(vv.packages):
    print(f"{i} {pkg[0]:10s} {pkg[1]:s}")

**** module numpy is installed
**** module pandas is installed
**** module matplotlib is installed
**** module scipy is installed
**** module astropy is installed
**** module astroquery is installed
**** module photutils is installed
**** module ysfitsutilpy is installed
**** module ysphotutilpy is installed
**** module ccdproc is installed
**** module aplpy is installed
**** module version_information is installed
This notebook was generated at 2024-07-04 19:13:22 (KST = GMT+0900) 
0 Python     3.12.3 64bit [GCC 11.2.0]
1 IPython    8.25.0
2 OS         Linux 5.15.0 107 generic x86_64 with glibc2.31
3 numpy      1.26.4
4 pandas     2.2.2
5 matplotlib 3.9.0
6 scipy      1.13.0
7 astropy    6.1.0
8 astroquery 0.4.7
9 photutils  1.12.0
10 ysfitsutilpy 0.2
11 ysphotutilpy 0.1.1
12 ccdproc    2.4.2
13 aplpy      2.1.0
14 version_information 1.0.4


### import modules

In [2]:
#%%
from glob import glob
from pathlib import Path
import os
import numpy as np
import matplotlib.pyplot as plt
from astropy.io import fits
import astropy.units as u
from astropy.stats import sigma_clip
from ccdproc import combine, ccd_process, CCDData
from astroquery.astrometry_net import AstrometryNet

import ysfitsutilpy as yfu

import _astro_utilities
import _Python_utilities
import _tool_visualization

plt.rcParams.update({'figure.max_open_warning': 0})

In [3]:
#%%
#######################################################
BASEDIR = Path("/mnt/Rdata/OBS_data") 
PROJECDIR = Path("/mnt/Rdata/OBS_data/2024-EXO")
TODODIR = PROJECDIR / "_-_-_2024-05_-_GSON300_STF-8300M_-_1bin"
TODODIR = PROJECDIR / "_-_-_2024-06_-_GSON300_STF-8300M_-_1bin"

PROJECDIR = Path("/mnt/Rdata/OBS_data/2022-Asteroid")
TODODIR = PROJECDIR / "GSON300_STF-8300M_-_1bin"
TODODIR = PROJECDIR / "RiLA600_STX-16803_-_1bin"
# TODODIR = PROJECDIR / "RiLA600_STX-16803_-_2bin"

# PROJECDIR = Path("/mnt/Rdata/OBS_data/2023-Asteroid")
# TODODIR = PROJECDIR / "GSON300_STF-8300M_-_1bin"
# TODODIR = PROJECDIR / "RiLA600_STX-16803_-_1bin"
# TODODIR = PROJECDIR / "RiLA600_STX-16803_-_2bin"

# PROJECDIR = Path("/mnt/Rdata/OBS_data/2016-Variable")
# TODODIR = PROJECDIR / "-_-_-_2016-_-_RiLA600_STX-16803_-_2bin"

PROJECDIR = Path("/mnt/Rdata/OBS_data/2017-Variable")
TODODIR = PROJECDIR / "-_-_-_2017-_-_RiLA600_STX-16803_-_2bin"

DOINGDIRs = sorted(_Python_utilities.getFullnameListOfsubDirs(TODODIR))
print ("DOINGDIRs: ", format(DOINGDIRs))
print ("len(DOINGDIRs): ", format(len(DOINGDIRs)))

CALDIR = [x for x in DOINGDIRs if "CAL-BDF" in str(x)]
MASTERDIR = Path(CALDIR[0]) / _astro_utilities.master_dir
if not MASTERDIR.exists():
    os.makedirs("{}".format(str(MASTERDIR)))
    print("{} is created...".format(str(MASTERDIR)))

print ("MASTERDIR: ", format(MASTERDIR))

DOINGDIRs = sorted([x for x in DOINGDIRs if "_LIGHT_" in str(x)])
print ("DOINGDIRs: ", format(DOINGDIRs))
print ("len(DOINGDIRs): ", format(len(DOINGDIRs)))

# filter_str = '2023-12-'
# DOINGDIRs = [x for x in DOINGDIRs if filter_str in x]
# remove = 'BIAS'
# DOINGDIRs = [x for x in DOINGDIRs if remove not in x]
# remove = 'DARK'
# DOINGDIRs = [x for x in DOINGDIRs if remove not in x]
# remove = 'FLAT'
# DOINGDIRs = [x for x in DOINGDIRs if remove not in x]
print ("DOINGDIRs: ", DOINGDIRs)
print ("len(DOINGDIRs): ", len(DOINGDIRs))
#######################################################

DOINGDIRs:  ['/mnt/Rdata/OBS_data/2017-Variable/-_-_-_2017-_-_RiLA600_STX-16803_-_2bin/-_CAL-BDF_-_2017-06_-_RiLA600_STX-16803_-_2bin/', '/mnt/Rdata/OBS_data/2017-Variable/-_-_-_2017-_-_RiLA600_STX-16803_-_2bin/V1257-HER_LIGHT_-_2017-06-04_-_RiLA600_STX-16803_-_2bin/', '/mnt/Rdata/OBS_data/2017-Variable/-_-_-_2017-_-_RiLA600_STX-16803_-_2bin/V1257-HER_LIGHT_-_2017-06-11_-_RiLA600_STX-16803_-_2bin/', '/mnt/Rdata/OBS_data/2017-Variable/-_-_-_2017-_-_RiLA600_STX-16803_-_2bin/V1257-HER_LIGHT_-_2017-06-14_-_RiLA600_STX-16803_-_2bin/', '/mnt/Rdata/OBS_data/2017-Variable/-_-_-_2017-_-_RiLA600_STX-16803_-_2bin/V1257-HER_LIGHT_-_2017-06-15_-_RiLA600_STX-16803_-_2bin/', '/mnt/Rdata/OBS_data/2017-Variable/-_-_-_2017-_-_RiLA600_STX-16803_-_2bin/V1257-HER_LIGHT_-_2017-06-16_-_RiLA600_STX-16803_-_2bin/', '/mnt/Rdata/OBS_data/2017-Variable/-_-_-_2017-_-_RiLA600_STX-16803_-_2bin/V1257-HER_LIGHT_-_2017-06-17_-_RiLA600_STX-16803_-_2bin/', '/mnt/Rdata/OBS_data/2017-Variable/-_-_-_2017-_-_RiLA600_STX-1680

In [6]:
#%%
ast = AstrometryNet()

# ger from nova.astrometry.net
ast.api_key = 'bldvwzzuvktnwfph' #must changed...

In [7]:
#%%
for DOINGDIR in DOINGDIRs[:] :
    DOINGDIR = Path(DOINGDIR)
    print("DOINGDIR", DOINGDIR)
    if "RiLA600_STX-16803_" in str(DOINGDIR.parts[-2]) :
        DOINGDIR = DOINGDIR / _astro_utilities.reduced_nightsky_dir
    if "GSON300_STF-8300M_" in str(DOINGDIR.parts[-2]) :
        DOINGDIR = DOINGDIR / _astro_utilities.reduced_dir

In [8]:
DOINGDIR = Path(DOINGDIRs[0])
print("DOINGDIR", DOINGDIR)
if "RiLA600_STX-16803_" in str(DOINGDIR.parts[-2]) :
    DOINGDIR = DOINGDIR / _astro_utilities.reduced_nightsky_dir
if "GSON300_STF-8300M_" in str(DOINGDIR.parts[-2]) :
    DOINGDIR = DOINGDIR / _astro_utilities.reduced_dir

summary = yfu.make_summary(DOINGDIR/"*.fit*")
if summary is not None :
    print("len(summary):", len(summary))
    print("summary:", summary)
    #print(summary["file"][0])  
    df_light = summary.loc[summary["IMAGETYP"] == "LIGHT"].copy()
    df_light = df_light.reset_index(drop=True)
    print("df_light:\n{}".format(df_light))

    for _, row  in df_light.iterrows():
        fpath = Path(row["file"])
        print(fpath)
        hdul = fits.open(fpath)

        submission_id = None
        solve_timeout = 600

        if 'PIXSCALE' in hdul[0].header:
            PIXc = hdul[0].header['PIXSCALE']
        else : 
            PIXc = _astro_utilities.calPixScale(hdul[0].header['FOCALLEN'], 
                                                hdul[0].header['XPIXSZ'],
                                                hdul[0].header['XBINNING'])
        print("PIXc : ", PIXc)
        hdul.close()

        SOLVE, ASTAP, LOCAL = _astro_utilities.checkPSolve(fpath)
        print("SOLVE:", SOLVE, "ASTAP:", ASTAP, "LOCAL:", LOCAL)

        if SOLVE :
            print(f"{fpath.name} is already solved...")
        else :             
            try_again = True                

            try : 
                
                while try_again:
                    try:
                        if not submission_id:
                            wcs_header = ast.solve_from_image(str(fpath),
                                                force_image_upload=True,
                                                solve_timeout = solve_timeout,
                                                submission_id=submission_id)
                        else:
                            wcs_header = ast.monitor_submission(submission_id,
                                                                solve_timeout = solve_timeout)
                    except TimeoutError as e:
                        submission_id = e.args[1]
                    else:
                        # got a result, so terminate
                        try_again = False

                if not wcs_header:
                    # Code to execute when solve fails
                    print("fits file solving failure...")

                else:
                    # Code to execute when solve succeeds
                    print("fits file solved successfully...")

                    with fits.open(str(fpath), mode='update') as hdul:
                        for card in wcs_header :
                            try: 
                                print(card, wcs_header[card], wcs_header.comments[card])
                                hdul[0].header.set(card, wcs_header[card], wcs_header.comments[card])
                            except : 
                                print(card)
                        hdul.flush

                    print(str(fpath)+" is created...")
            
            except Exception as err: 
                print("Err :", err)
                continue 

MAIN_ID,RA,DEC,RA_PREC,DEC_PREC,COO_ERR_MAJA,COO_ERR_MINA,COO_ERR_ANGLE,COO_QUAL,COO_WAVELENGTH,COO_BIBCODE,SCRIPT_NUMBER_ID
,"""h:m:s""","""d:m:s""",,,mas,mas,deg,,,,
object,str13,str13,int16,int16,float32,float32,int16,str1,str1,object,int32
Qatar 1b,20 13 31.6172,+65 09 43.492,14,14,0.011,0.009,90,A,O,2020yCat.1350....0G,1
Qatar 1,20 13 31.6172,+65 09 43.492,14,14,0.011,0.009,90,A,O,2020yCat.1350....0G,1
